# データセットの準備
外れ値検知ではモデル学習と評価のために正常データと異常データからデータセットを作成します。

データセットには以下３種類のデータとMaskデータ含まれています。

* Trainデータ（訓練データ）
* Validationデータ
* Testデータ

上記以外に
* Maskデータ

### Trainデータ
Trainデータは、正常データだけで構成されます。
外れ値検知ではTrainデータの特徴を何らかの形でモデルが記憶する、ということになります。

### Validationデータ
Validationデータは通常学習中のモデル評価に使われますが、Anomalibの各手法では異なった使われ方をするケースがあります。
PatchCoreでは、出力するスコアの正規化パラメータ算出に使われます。
正常データ、異常データを両方含みます。

### Testデータ
最終的なモデル性能評価に使われます。そのため学習やチューニングでは一切使用しません。
正常データ、異常データを両方含みます。

### Maskデータ
正解画像として、異常ピクセルをラベリングした画像データになります。
ピクセル単位での評価に用います（画像単位での評価では使われません）

# Anomalibでのデータセット作成
Anomalibではいくつかデータセットを作成する方法がありますが、今回はデータ設定ファイル（.yaml）を用いてデータセットを作成してみます。

## 画像データをディレクトリに配置
まず正常データ、異常データをディレクトリに配置します。
今回は題材として木材の異常検知を扱います。

以下のディレクトリ構成で画像データを配置しました。

```
datasets/
  wood/
    test/
      NG/
        異常データ 36枚
      OK/
        正常データ 36枚
    test_gt/
      マスク画像 36枚
    train/
      OK/
        正常データ 230枚
      OK_14/
        正常データ 14枚（tain/OKから一部抜粋したもの）
      OK_35/
        正常データ 35枚（train/OKから一部抜粋したもの）
```

みたところValidation用のディレクトリがないように見えますが、AnomalibではValidationデータのディレクトリを指定するインターフェースが用意されていません。
ValidationデータはTestの画像を一部分割するなどして作る形となります。

※今回は実習環境がCPU環境のため学習時間を短縮するために、OK_14、OK_35という正常Trainデータのサブセットを用意しています。

# どんな画像があるか見てみます
## 正常画像

In [ ]:
from utils import get_image_path_list, show_images

paths = get_image_path_list("datasets/wood/train/OK", pathlib=False)

show_images(paths[:6])

## 異常画像・マスク画像

In [ ]:
from utils import get_image_path_list, show_multi_images

ng_paths = get_image_path_list("datasets/wood/test/NG", pathlib=False)
mask_paths = get_image_path_list("datasets/wood/test_gt", pathlib=False)

show_multi_images([ng_paths[:6], mask_paths[:6]])

## データセットの設定設定ファイル xxxx.yaml
上記ディレクトリ構成の画像ファイルたちをどのようにTrain,Validation, Testデータに振り分けるかをxxxx.yamlで設定します。

今回は木材データセットなので、名前をwoodデータセットとします。設定ファイル名はwood.yamlとします。

wood.yamlの内容は以下となります。

```yaml:wood.yaml
data:
  class_path: anomalib.data.Folder
  init_args:
    name: "wood"
    root: "datasets/wood"
    normal_dir: "train/OK_14"
    abnormal_dir: "test/NG"
    normal_test_dir: "test/OK"
    mask_dir: "test_gt"
    extensions: [".png", ".jpg", ".jpeg", ".bmp"]

    train_batch_size: 32
    eval_batch_size: 32
    num_workers: 0
    test_split_mode: from_dir
    val_split_mode: from_test
    val_split_ratio: 0.5
    seed: 777
```

![img](./img/cap_dataset_divide.png)

以下はディレクトリを指定するパラメータになります
* normal_dir: Train正常データのディレクトリパス
* abnromal_dir: Test異常データのディレクトリパス
* normal_test_dir: Test正常データのディレクトリパス

rootは画像データのルートパスを意味し、rootがdatasets/woodの場合は以下となります。
* Train正常データ→ datasets/wood/train/OK_14
* Test異常データ→ datasets/wood/test/NG
* Test正常データ→ datasets/wood/test/OK

データの分割は上図のように行います
* test_split_mode = from_dirは、Testデータにabnormal_dirとnormal_test_dirで指定されたデータを使うことを示す
* val_split_mode = same_as_testは、Testデータの一部をValidationデータとして使うことを示す
* val_split_ratioは、Testデータの何割をValidationデータとして使うかを示す

正常・異常は偏りがないように分割されます。

他のパラメータは
* name: データセット名
* extensions: 対象とするファイルの拡張子
* train_batch_size: 学習バッチサイズ
* eval_batch_size: 評価バッチサイズ
* num_workers: データ読み込みを並列化するためのワーカープロセス数　※Windowsでは問題が起きやすいため、0にすることが多い
* seed: 乱数シード。ファイルの分割がある場合分割ファイルはランダムで選択される。固定したい場合はこのパラメータを設定する

# モデル設定ファイルの準備
今回PatchCoreを使うが、モデルの設定ファイルは以下となる。

```yaml:patchcore.yaml
model:
  class_path: anomalib.models.Patchcore
  init_args:
    backbone: wide_resnet50_2
    layers:
      - layer2
      - layer3
    pre_trained: true
    coreset_sampling_ratio: 0.1
    num_neighbors: 9
     
```

モデルの設定ファイルは基本下記から持ってきてそのまま使うで問題ありません。

https://github.com/open-edge-platform/anomalib/tree/v2.2.0/examples/configs/model

